# 🩺 EDA de dataset real: Cardiovascular Disease (Kaggle)

Este notebook realiza un **análisis exploratorio de datos (EDA)** sobre el dataset real de **enfermedad cardiovascular** publicado en Kaggle ("Cardiovascular Disease Dataset").

El objetivo es:

- Entender la distribución de las variables clínicas y de estilo de vida.
- Limpiar valores extremos/extraños (por ejemplo, presión arterial imposible).
- Estimar la **probabilidad de enfermedad cardiovascular / hipertensión** por grupos de edad, IMC, sexo, tabaquismo, etc.
- Conectar estos hallazgos con el proyecto de **clasificación de hipertensión** que ya construiste (modelo + API + dashboard).

> ⚠️ Nota: Este dataset contiene datos reales anonimizados. El análisis es educativo y no sustituye la opinión de especialistas médicos.


## 1. Carga de librerías y datos

Antes de ejecutar este notebook, asegúrate de que el archivo de Kaggle esté descargado y descomprimido en:

```text
data/real/cardio/cardio_train.csv
```

(El nombre puede variar; ajusta la ruta si tu archivo tiene otro nombre).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# Carga del dataset (ajusta la ruta si es necesario)
df = pd.read_csv("data/real/cardio/cardio_train.csv", sep=';')

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/real/cardio/cardio_train.csv'

## 2. Exploración inicial y estructura del dataset

In [ ]:
df.info()

In [ ]:
df.describe().T

### Diccionario básico de variables

Según la documentación de Kaggle, las columnas típicas son (puede variar ligeramente):

- `age`: edad en **días**
- `gender`: 1 = Mujer, 2 = Hombre (en algunas versiones)
- `height`: altura en cm
- `weight`: peso en kg
- `ap_hi`: presión arterial sistólica (mmHg)
- `ap_lo`: presión arterial diastólica (mmHg)
- `cholesterol`: 1 = normal, 2 = por encima de lo normal, 3 = muy por encima
- `gluc`: 1 = normal, 2 = por encima de lo normal, 3 = muy por encima
- `smoke`: 0 = no fuma, 1 = fuma
- `alco`: 0 = no bebe, 1 = consume alcohol con frecuencia
- `active`: 0 = físicamente inactivo, 1 = activo
- `cardio`: 0 = sin enfermedad cardiovascular, 1 = con enfermedad cardiovascular


## 3. Limpieza básica y creación de variables derivadas

In [ ]:
# Copia para trabajar
df_clean = df.copy()

# 1) Convertir edad de días a años
df_clean["age_years"] = (df_clean["age"] / 365.25).round(1)

# 2) Crear IMC
df_clean["BMI"] = df_clean["weight"] / (df_clean["height"]/100)**2

# 3) Corregir codificación de género (si aplica)
# En algunas versiones: 1 = Mujer, 2 = Hombre
if set(df_clean["gender"].unique()) == {1, 2}:
    df_clean["gender_text"] = df_clean["gender"].map({1: "Femenino", 2: "Masculino"})
else:
    df_clean["gender_text"] = df_clean["gender"]

# 4) Filtrar valores de presión arterial imposibles
mask_bp = (df_clean["ap_hi"].between(80, 260)) & (df_clean["ap_lo"].between(40, 150)) & (df_clean["ap_hi"] >= df_clean["ap_lo"])
df_clean = df_clean[mask_bp].copy()

# 5) Opcional: filtrar IMC en rango razonable
mask_bmi = df_clean["BMI"].between(10, 60)
df_clean = df_clean[mask_bmi].copy()

df_clean[["age_years", "BMI", "ap_hi", "ap_lo"]].describe().T

## 4. Distribución de la variable objetivo `cardio`

In [ ]:
ax = sns.countplot(x="cardio", data=df_clean)
plt.xlabel("Presencia de enfermedad cardiovascular (cardio)")
plt.ylabel("Número de pacientes")
plt.title("Distribución de la variable objetivo (cardio)")
plt.xticks([0, 1], ["Sin enfermedad", "Con enfermedad"])
plt.tight_layout()
plt.show()

df_clean["cardio"].value_counts(normalize=True).rename(index={0: "Sin enfermedad", 1: "Con enfermedad"})

## 5. Variable binaria: ¿Hipertenso sí/no? (a partir de la presión arterial)

Aunque el dataset tiene la variable `cardio` como indicador general de enfermedad cardiovascular, podemos definir una variable aproximada de **hipertensión** a partir de los valores de `ap_hi` y `ap_lo`.

Una regla muy usada (simplificada) es considerar hipertenso si:

- `ap_hi ≥ 140` **o**
- `ap_lo ≥ 90`

In [ ]:
df_clean["hipertenso_bp"] = (
    (df_clean["ap_hi"] >= 140) | (df_clean["ap_lo"] >= 90)
).astype(int)

df_clean["hipertenso_bp"].value_counts(normalize=True).rename(index={0: "No hipertenso", 1: "Hipertenso"})

## 6. Probabilidad de hipertensión por grupos de edad

In [ ]:
bins_age = [18, 30, 45, 60, 90]
labels_age = ["18-29", "30-44", "45-59", "60+"]

df_clean["age_group"] = pd.cut(df_clean["age_years"], bins=bins_age, labels=labels_age, right=False)

prob_age = df_clean.groupby("age_group")["hipertenso_bp"].mean().rename("prob_hipertension")
display(prob_age)

ax = prob_age.mul(100).plot(kind="bar")
plt.ylabel("Probabilidad de hipertensión (%)")
plt.title("Probabilidad de hipertensión por grupo de edad")
plt.ylim(0, 100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Probabilidad de hipertensión por categoría de IMC

In [ ]:
def bmi_category(bmi):
    if bmi < 18.5:
        return "Bajo peso"
    elif bmi < 25:
        return "Normal"
    elif bmi < 30:
        return "Sobrepeso"
    else:
        return "Obesidad"

df_clean["BMI_cat"] = df_clean["BMI"].apply(bmi_category)

prob_bmi = df_clean.groupby("BMI_cat")["hipertenso_bp"].mean().rename("prob_hipertension")
prob_bmi = prob_bmi.reindex(["Bajo peso", "Normal", "Sobrepeso", "Obesidad"])

display(prob_bmi)

ax = prob_bmi.mul(100).plot(kind="bar")
plt.ylabel("Probabilidad de hipertensión (%)")
plt.title("Probabilidad de hipertensión por categoría de IMC")
plt.ylim(0, 100)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 8. Probabilidad de hipertensión por sexo

In [ ]:
prob_sex = df_clean.groupby("gender_text")["hipertenso_bp"].mean().rename("prob_hipertension")
display(prob_sex)

ax = prob_sex.mul(100).plot(kind="bar")
plt.ylabel("Probabilidad de hipertensión (%)")
plt.title("Probabilidad de hipertensión por sexo")
plt.ylim(0, 100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Factores de estilo de vida: tabaquismo, alcohol y actividad física

In [ ]:
df_clean["smoke_text"] = df_clean["smoke"].map({0: "No fumador", 1: "Fumador"})
df_clean["alco_text"] = df_clean["alco"].map({0: "No bebe frecuentemente", 1: "Consume alcohol"})
df_clean["active_text"] = df_clean["active"].map({0: "Inactivo", 1: "Activo"})

prob_smoke = df_clean.groupby("smoke_text")["hipertenso_bp"].mean().rename("prob_hipertension")
prob_alco = df_clean.groupby("alco_text")["hipertenso_bp"].mean().rename("prob_hipertension")
prob_active = df_clean.groupby("active_text")["hipertenso_bp"].mean().rename("prob_hipertension")

display(prob_smoke)
display(prob_alco)
display(prob_active)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

prob_smoke.mul(100).plot(kind="bar", ax=axes[0])
axes[0].set_title("Hipertensión según tabaquismo")
axes[0].set_ylim(0, 100)

prob_alco.mul(100).plot(kind="bar", ax=axes[1])
axes[1].set_title("Hipertensión según consumo de alcohol")
axes[1].set_ylim(0, 100)

prob_active.mul(100).plot(kind="bar", ax=axes[2])
axes[2].set_title("Hipertensión según actividad física")
axes[2].set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 10. Vista conjunta: Edad, IMC e hipertensión

In [ ]:
sample_n = min(5000, len(df_clean))
df_sample = df_clean.sample(sample_n, random_state=42)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_sample,
    x="age_years",
    y="BMI",
    hue="hipertenso_bp",
    palette={0: "tab:blue", 1: "tab:red"},
    alpha=0.5
)
plt.xlabel("Edad (años)")
plt.ylabel("IMC")
plt.title("Edad vs IMC coloreado por hipertensión (según presión arterial)")
plt.tight_layout()
plt.show()

## 11. Conclusiones y relación con tu modelo / app

De este análisis sobre el dataset real de enfermedad cardiovascular se pueden sacar ideas como:

- La **prevalencia de hipertensión** (definida a partir de `ap_hi` / `ap_lo`) en esta población real.
- Cómo la probabilidad de hipertensión **aumenta con la edad** y con categorías más altas de **IMC**.
- Diferencias por **sexo** y por factores de estilo de vida como **tabaquismo, alcohol y actividad física**.

### Conexión con tu proyecto de hipertensión

- Puedes usar este dataset real para **reentrenar** o **validar** tu modelo (Random Forest / XGBoost / ANN) entrenado inicialmente con datos sintéticos.
- Las probabilidades por grupo pueden ayudarte a:
  - Ajustar mensajes de riesgo en tu dashboard o chatbot.
  - Comparar resultados entre el dataset sintético y el real.
- En un trabajo formal, podrías discutir **limitaciones** (definición de hipertensión, auto-reporte, posibles sesgos) y cómo afectaría a la generalización del modelo.

Este notebook es un buen punto de partida para integrar datos reales a tu pipeline de ML en hipertensión arterial.
